# Customer Segmentation for a Retail Business using K-Means Clustering
**SOEN 400: Data Mining & Analytics — Group 3**

## 1. Project Overview
A UK online retailer has thousands of customers who buy in different ways. The data contain no customer groups, so we use **K-Means clustering** to find groups of customers with similar purchasing behaviour.

**Steps:** load and examine the data → check data quality → clean → build one row per customer (RFM) → explore RFM → transform and scale → choose K → train K-Means → profile, evaluate and interpret the clusters.

The notebook needs only `Online Retail.csv` (UCI Online Retail dataset) in the same folder. Run all cells from top to bottom; results are saved to `outputs/`.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

RANDOM_STATE = 42                     # same result every time the notebook runs
OUT_DIR = Path('outputs')
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

RFM_COLS = ['Recency', 'Frequency', 'Monetary']
COLORS = ['#2a78d6', '#eb6834', '#1baf7a']    # Cluster 1, 2, 3

def save_fig(fig, name):
    fig.savefig(FIG_DIR / name, dpi=200, bbox_inches='tight', facecolor='white')

## 2. Load and Examine the Dataset
We look at the size of the data, its columns and data types, and a few example rows.

In [ ]:
raw = pd.read_csv('Online Retail.csv')
print(f'Rows: {len(raw):,}   Columns: {raw.shape[1]}')
print(f"Invoices: {raw['InvoiceNo'].nunique():,}   Customer IDs: {raw['CustomerID'].nunique():,}")

display(pd.DataFrame({'Data type': raw.dtypes.astype(str), 'Non-null values': raw.notna().sum(),
                      'Unique values': raw.nunique()}))
display(raw.head())
display(raw.describe().T)

The data are **transaction-level**: each row is one product line on an invoice, so one invoice has many rows and one customer has many invoices. To group customers we must later turn this into **one row per customer**.

## 3. Data Quality Analysis
### 3.1 Missing values

In [ ]:
missing = pd.DataFrame({'Missing rows': raw.isna().sum(), '% of rows': raw.isna().mean() * 100})
missing = missing[missing['Missing rows'] > 0]
display(missing)

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(missing.index, missing['% of rows'], color=COLORS[0])
ax.bar_label(bars, fmt='%.2f%%')
ax.set(title='Missing values per column', ylabel='% of rows')
fig.tight_layout(); plt.show()

About a quarter of the rows have no `CustomerID`. These purchases cannot be linked to a customer, so they cannot be used for customer segmentation. Missing product descriptions are rare and are not needed for RFM.

### 3.2 Duplicates, cancellations and invalid values

In [ ]:
invoice = raw['InvoiceNo'].astype(str)
issues = pd.Series({
    'Missing CustomerID': raw['CustomerID'].isna().sum(),
    'Cancelled invoices (start with C)': invoice.str.startswith('C').sum(),
    'Quantity <= 0': (raw['Quantity'] <= 0).sum(),
    'UnitPrice <= 0': (raw['UnitPrice'] <= 0).sum(),
    'Exact duplicate rows': raw.duplicated().sum(),
})
quality = pd.DataFrame({'Rows': issues, '% of rows': issues / len(raw) * 100})
display(quality)

dates = pd.to_datetime(raw['InvoiceDate'], format='%m/%d/%Y %H:%M:%S')
print(f'Date range: {dates.min().date()} to {dates.max().date()}')

fig, ax = plt.subplots(figsize=(8, 3.8))
ordered = issues.sort_values()
bars = ax.barh(ordered.index, ordered.values, color=COLORS[1])
ax.bar_label(bars, labels=[f'{v:,}' for v in ordered.values], padding=3)
ax.set(title='Data-quality issues (rows)', xlabel='Rows')
ax.margins(x=0.15)
fig.tight_layout(); plt.show()

These issues overlap: for example, every cancelled invoice line also has a negative quantity. Negative quantities can be genuine returns, and the data do not explain why some prices are zero, so these rows are not automatically "errors" – but they are not positive purchases.

### 3.3 Transactions over time

In [ ]:
monthly = dates.dt.to_period('M').value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index.astype(str), monthly.values, marker='o', color=COLORS[0])
ax.set(title='Transaction lines per month', ylabel='Rows', xlabel='Month')
plt.xticks(rotation=45, ha='right')
fig.tight_layout(); plt.show()

Activity rises in the later months of 2011 and peaks around November. December 2011 looks low only because the data end on 9 December.

## 4. Data Cleaning
We keep only valid **positive purchases** that can be linked to a customer:
* remove rows without a `CustomerID` (cannot be linked to a customer);
* remove cancelled invoices;
* keep `Quantity > 0` and `UnitPrice > 0` (positive purchases only);
* remove exact duplicate rows (they would count purchases twice).

Because the issues overlap, a step can remove fewer rows than the counts in 3.2.

In [ ]:
steps = [('Raw data', len(raw))]
df = raw.dropna(subset=['CustomerID'])                            # Remove rows without a customer ID
steps.append(('Remove missing CustomerID', len(df)))
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]         # Remove cancelled invoices
steps.append(('Remove cancelled invoices', len(df)))
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]             # Keep positive purchases only
steps.append(('Keep Quantity > 0 and UnitPrice > 0', len(df)))
df = df.drop_duplicates()                                         # Remove exact duplicates
steps.append(('Remove exact duplicates', len(df)))

cleaning = pd.DataFrame(steps, columns=['Step', 'Rows remaining'])
cleaning['Rows removed'] = (-cleaning['Rows remaining'].diff()).fillna(0).astype(int)
display(cleaning)
print(f'Retained {len(df):,} of {len(raw):,} rows = {len(df) / len(raw) * 100:.2f}% of the original records')

## 5. Build Customer-Level RFM
We want to group **customers**, not transaction lines, so we make one row per customer with three features:
* **Recency** – how recently the customer purchased (days since the last purchase; lower = more recent).
* **Frequency** – how often the customer purchased (number of distinct invoices).
* **Monetary** – how much the customer spent (total of `Quantity × UnitPrice` over the retained purchases).

Recency is measured from one day after the last invoice in the data. Monetary is gross purchase value (returns and cancellations were excluded).

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%Y %H:%M:%S')
df['CustomerID'] = df['CustomerID'].astype(int)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']               # Calculate purchase value

snapshot = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda d: (snapshot - d.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum'),
).reset_index()

print(f'Snapshot date: {snapshot.date()}   Customers: {len(rfm):,}')
summary = rfm[RFM_COLS].describe().T
summary['skewness'] = rfm[RFM_COLS].skew()
display(summary)

## 6. Explore the RFM Features
### 6.1 Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, RFM_COLS):
    ax.hist(rfm[col], bins=50, color=COLORS[0])
    ax.axvline(rfm[col].median(), color='black', ls='--', lw=1.2, label=f'median = {rfm[col].median():,.0f}')
    ax.set(title=f'{col} (skew = {rfm[col].skew():.2f})', xlabel=col, ylabel='Customers')
    ax.legend()
fig.tight_layout(); plt.show()

Most customers buy a few times and spend modestly, but a small number buy very often or spend very large amounts. Frequency and Monetary are therefore highly right-skewed.

### 6.2 Outliers
Outliers are counted with the IQR rule (below Q1 − 1.5·IQR or above Q3 + 1.5·IQR).

In [ ]:
rows = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, RFM_COLS):
    q1, q3 = rfm[col].quantile([0.25, 0.75])
    lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    n_out = ((rfm[col] < lower) | (rfm[col] > upper)).sum()
    rows.append({'Feature': col, 'Lower limit': lower, 'Upper limit': upper,
                 'Outlier customers': n_out, '% of customers': n_out / len(rfm) * 100})
    ax.boxplot(rfm[col], patch_artist=True)
    ax.set(title=f'{col} – {n_out:,} outliers', ylabel=col)
fig.tight_layout(); plt.show()
display(pd.DataFrame(rows).set_index('Feature'))

We **keep** these customers. Very high spending or frequency can be genuine behaviour – these may be the most valuable customers. Instead of deleting them, we reduce their influence with a log transformation (section 7).

### 6.3 Relationships between the features

In [ ]:
display(rfm[RFM_COLS].corr().style.format('{:.3f}'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(rfm[RFM_COLS].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, ax=axes[0])
axes[0].set_title('Correlation between RFM features')
axes[1].scatter(rfm['Frequency'], rfm['Monetary'], s=10, alpha=0.3, color=COLORS[0])
axes[1].set(xscale='log', yscale='log', title='Frequency vs Monetary (log scales)', xlabel='Frequency', ylabel='Monetary (£)')
fig.tight_layout(); plt.show()

Frequency and Monetary have a **moderate positive** relationship: customers who buy more often tend to spend more in total. Recency is only weakly (negatively) related to the other two. Each feature therefore adds different information about a customer.

## 7. Transform and Scale the Features
K-Means groups customers using distances, so the features must be comparable. The table below shows how different their ranges are.

In [ ]:
display(pd.DataFrame({'min': rfm[RFM_COLS].min(), 'max': rfm[RFM_COLS].max(),
                      'range': rfm[RFM_COLS].max() - rfm[RFM_COLS].min()}))

* **Log transform:** `log1p(x) = ln(1 + x)` reduces the long tail of Frequency and Monetary, so a few extreme customers do not pull the clusters towards them. Recency is only moderately skewed and stays in days.
* **Standardization:** Monetary's range is far larger than the others, so it would dominate the distances. Scaling each feature to mean 0 and standard deviation 1 makes the three features comparable.

In [ ]:
features = rfm[RFM_COLS].copy()
features['Frequency'] = np.log1p(features['Frequency'])          # Transform Frequency and Monetary
features['Monetary'] = np.log1p(features['Monetary'])
X = StandardScaler().fit_transform(features)                      # Standardize the RFM features

display(pd.DataFrame({'skew before': rfm[RFM_COLS].skew(), 'skew after log1p': features.skew(),
                      'mean after scaling': X.mean(axis=0), 'std after scaling': X.std(axis=0)}))

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for j, col in enumerate(RFM_COLS):
    axes[0, j].hist(rfm[col], bins=50, color=COLORS[0])
    axes[0, j].set_title(f'{col} – original')
    axes[1, j].hist(features[col], bins=50, color=COLORS[2])
    axes[1, j].set_title(f'{col} – ' + ('unchanged' if col == 'Recency' else 'after log1p'))
fig.suptitle('RFM features before and after transformation', fontsize=14)
fig.tight_layout(); save_fig(fig, '01_rfm_before_after_log.png'); plt.show()

## 8. How K-Means Works
K-Means groups the 4,338 customers using their three scaled RFM features.

**K-Means (Lloyd's algorithm)**
1. **Choose K** – the number of clusters.
2. **Initialise** – pick K starting centroids.
3. **Assignment step** – compute the distance from every customer to every centroid and assign each customer to the *nearest* centroid.
4. **Update step** – move each centroid to the *mean* of the customers assigned to it.
5. **Iterate** – repeat steps 3–4.
6. **Converge / stop** – when no customer changes cluster (equivalently, the centroids stop moving), or a maximum number of iterations is reached.

The objective K-Means minimises is the **within-cluster sum of squares (WCSS, "inertia")**: $\sum_{k}\sum_{x \in C_k} \lVert x - \mu_k \rVert^2$. Each assignment step and each update step can only lower (or keep) this value, which is why the algorithm converges.

## 9. Select the Number of Clusters (K)
We try **K = 2 to 10** and compare two measures:
* **Elbow method:** inertia (WCSS) always falls as K grows; we look for the point where adding clusters stops helping much.
* **Silhouette score:** shows how well customers fit their own cluster compared with the nearest other cluster (from −1 to 1; higher is better).

In [ ]:
rows = []
for k in range(2, 11):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit(X)
    rows.append({'K': k, 'inertia': model.inertia_, 'silhouette': silhouette_score(X, model.labels_)})
k_scores = pd.DataFrame(rows)
k_scores['inertia drop (%)'] = -k_scores['inertia'].pct_change() * 100
display(k_scores.style.format({'inertia': '{:,.1f}', 'silhouette': '{:.3f}', 'inertia drop (%)': '{:.1f}'}, na_rep='–').hide(axis='index'))
k_scores.to_csv(OUT_DIR / 'k_selection.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(k_scores['K'], k_scores['inertia'], marker='o', lw=2, color=COLORS[0])
ax.set(title='Elbow method', xlabel='Number of clusters K', ylabel='Inertia')
fig.tight_layout(); save_fig(fig, '02_elbow.png'); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(k_scores['K'], k_scores['silhouette'], marker='o', lw=2, color=COLORS[1])
for k, s in zip(k_scores['K'], k_scores['silhouette']):
    ax.annotate(f'{s:.3f}', (k, s), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax.set(title='Silhouette score by K', xlabel='Number of clusters K', ylabel='Mean silhouette score')
ax.margins(y=0.15)
fig.tight_layout(); save_fig(fig, '03_silhouette.png'); plt.show()

**Choice of K:** the inertia curve bends around K = 3–4 (after that, extra clusters help much less), and the silhouette score is highest at **K = 3**. K = 3 also gives three groups that are easy to describe, so we use **K = 3**. This is a reasoned choice, not proof that exactly three customer types exist.

## 10. Train the K-Means Model
Settings: `k-means++` places the starting centroids far apart, `n_init=20` runs the algorithm 20 times and keeps the result with the lowest inertia, and `random_state` makes the result reproducible.

In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', n_init=20, random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X)                                    # Fit the final K-Means model

# Number clusters 1-3 from highest to lowest average spending
order = rfm.groupby(labels)['Monetary'].mean().sort_values(ascending=False).index
rfm['Cluster'] = pd.Series(labels).map({old: new for new, old in enumerate(order, start=1)})

print(f'Converged after {kmeans.n_iter_} iterations')
rfm.to_csv(OUT_DIR / 'customer_segments.csv', index=False)

## 11. Cluster Profiles
The profile uses the original units (days, invoices, £). **Medians** describe the typical customer better than means because a few very large customers raise the means.

In [ ]:
g = rfm.groupby('Cluster')
profile = pd.DataFrame({
    'Customers': g.size(),
    '% of customers': g.size() / len(rfm) * 100,
    'Median Recency (days)': g['Recency'].median(),
    'Median Frequency (invoices)': g['Frequency'].median(),
    'Median Monetary (£)': g['Monetary'].median(),
    '% bought only once': g['Frequency'].apply(lambda s: (s == 1).mean() * 100),
    '% of total Monetary': g['Monetary'].sum() / rfm['Monetary'].sum() * 100,
})
profile.loc['All customers'] = [len(rfm), 100, rfm['Recency'].median(), rfm['Frequency'].median(),
                                rfm['Monetary'].median(), (rfm['Frequency'] == 1).mean() * 100, 100]
profile['Customers'] = profile['Customers'].astype(int)
display(profile)
profile.round(2).to_csv(OUT_DIR / 'cluster_profile.csv')

In [ ]:
sizes = rfm['Cluster'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar([f'Cluster {c}' for c in sizes.index], sizes.values, color=COLORS, width=0.6)
ax.bar_label(bars, labels=[f'{v:,} ({v / len(rfm):.1%})' for v in sizes.values], padding=3)
ax.set(title='Customers in each cluster', ylabel='Customers', ylim=(0, sizes.max() * 1.15))
fig.tight_layout(); save_fig(fig, '04_cluster_sizes.png'); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, col, unit in zip(axes, RFM_COLS, ['days since last purchase', 'invoices (log scale)', '£ spent (log scale)']):
    sns.boxplot(data=rfm, x='Cluster', y=col, hue='Cluster', palette=COLORS, legend=False, ax=ax, fliersize=2)
    if col != 'Recency':
        ax.set_yscale('log')
    ax.set(title=col, ylabel=unit)
fig.suptitle('Recency, Frequency and Monetary by cluster', fontsize=14)
fig.tight_layout(); save_fig(fig, '05_rfm_by_cluster.png'); plt.show()

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for c in sizes.index:
    d = rfm[rfm['Cluster'] == c]
    ax.scatter(d['Recency'], d['Monetary'], s=9, alpha=0.5, color=COLORS[c - 1], label=f'Cluster {c}')
ax.set_yscale('log')
ax.set(title='Customers by Recency and Monetary, coloured by cluster',
       xlabel='Recency (days since last purchase)', ylabel='Monetary (£, log scale)')
ax.legend(markerscale=2.5)
fig.tight_layout(); save_fig(fig, '06_recency_vs_monetary.png'); plt.show()

The scatter plot shows two of the three features; the clusters were formed using all three.

## 12. Basic Evaluation
We use the two measures from section 9: the elbow plot supported K = 3, and the silhouette score of the final model indicates how well customers fit their clusters.

In [ ]:
print(f'Inertia of the final model: {kmeans.inertia_:,.1f}')
print(f'Mean silhouette score:      {silhouette_score(X, labels):.3f}')

A silhouette score around 0.4 means the clusters are **reasonably but not perfectly separated**: customer behaviour changes gradually, so some customers sit near the boundary between two clusters. The clusters still differ clearly in their medians (section 11). There are no true customer labels, so no score can prove that these are the "correct" groups.

## 13. Interpretation and Business Recommendations
The labels below are **our description of each cluster after looking at its profile**; K-Means only produced the cluster numbers.

| Cluster | What the profile shows | Label | Possible action |
|---|---|---|---|
| 1 | Most recent purchases, many invoices, highest spending; most of the total purchase value | **Active, frequent, high-value** | Retention and loyalty benefits; watch for customers whose recency starts to rise |
| 2 | Bought fairly recently, usually one or two invoices, modest spending | **Recent, low-frequency, lower-value** | Follow-up offers that encourage a repeat purchase |
| 3 | Last purchase long ago, mostly a single invoice, lowest spending | **Lapsed, mostly one-time, low-value** | A carefully targeted, low-cost re-activation campaign, if the cost is justified |

These actions are suggestions based on the profiles; they have not been tested.